# 💻Clase 16 - Introducción a Dataframes

---

# Sesión 1 :

In [91]:
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.WARN)
Logger.getLogger("akka").setLevel(Level.WARN)

import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Spark411-Almond")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

println(s"Java: ${System.getProperty("java.version")}")
println(s"Spark: ${spark.version}")
println(s"Scala: ${scala.util.Properties.versionNumberString}")

26/04/28 13:22:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Java: 17.0.18
Spark: 4.1.1
Scala: 2.13.18


import org.apache.log4j.{Level, Logger}
import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@761b51a9
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@6e805d82

---

# 📦 Visualización de Particiones + Ejercicios `combineByKey`

---

# 🔍 Función para ver datos por partición

In [92]:
def verParticiones[T](rdd: org.apache.spark.rdd.RDD[T]): Unit = {
  rdd.mapPartitionsWithIndex { (index, iter) =>
    Iterator(s"📦 Partición $index -> ${iter.toList.mkString(", ")}")
  }.collect().foreach(println)
}

defined function verParticiones

---

# 🧪 Ejercicio 1: Temperaturas por ciudad

## 📥 Datos

In [93]:
val temperaturas = sc.parallelize(List(
  ("Madrid", 18.5),
  ("Barcelona", 20.0),
  ("Madrid", 21.0),
  ("Valencia", 22.5),
  ("Sevilla", 25.0),
  ("Barcelona", 19.5),
  ("Madrid", 17.0),
  ("Valencia", 23.0),
  ("Sevilla", 26.5),
  ("Barcelona", 18.0),
  ("Madrid", 20.5),
  ("Valencia", 21.5)
), 3)

verParticiones(temperaturas)

📦 Partición 0 -> (Madrid,18.5), (Barcelona,20.0), (Madrid,21.0), (Valencia,22.5)
📦 Partición 1 -> (Sevilla,25.0), (Barcelona,19.5), (Madrid,17.0), (Valencia,23.0)
📦 Partición 2 -> (Sevilla,26.5), (Barcelona,18.0), (Madrid,20.5), (Valencia,21.5)


temperaturas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[133] at parallelize at cmd93.sc:1

In [94]:
val temperaturasPorCiudad = temperaturas.combineByKey(
    (t: Double) => List(t),
    (acc: List[Double], t: Double) => acc :+ t,
    (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2
)

temperaturasPorCiudad.collect().sortBy(_._1).foreach { case (ciudad, temperatura) =>
    println(s"$ciudad : $temperatura")
}

Barcelona : List(20.0, 19.5, 18.0)
Madrid : List(18.5, 21.0, 17.0, 20.5)
Sevilla : List(25.0, 26.5)
Valencia : List(22.5, 23.0, 21.5)


temperaturasPorCiudad: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[135] at combineByKey at cmd94.sc:4

## 🎯 Objetivo

Agrupar las temperaturas registradas por ciudad.

## ✅ Salida esperada (aproximada)

```
(Madrid,List(18.5, 21.0, 17.0, 20.5))
(Barcelona,List(20.0, 19.5, 18.0))
(Valencia,List(22.5, 23.0, 21.5))
(Sevilla,List(25.0, 26.5))
```

---

# 🧪 Ejercicio 2: Productos comprados por cliente

## 📥 Datos

In [95]:
val compras = sc.parallelize(List(
  ("Ana", "Laptop"),
  ("Luis", "Teclado"),
  ("Ana", "Mouse"),
  ("Marta", "Monitor"),
  ("Luis", "Tablet"),
  ("Ana", "Auriculares"),
  ("Carlos", "Impresora"),
  ("Marta", "Webcam"),
  ("Luis", "Ratón"),
  ("Carlos", "SSD"),
  ("Ana", "USB"),
  ("Marta", "Silla")
), 3)

verParticiones(compras)

📦 Partición 0 -> (Ana,Laptop), (Luis,Teclado), (Ana,Mouse), (Marta,Monitor)
📦 Partición 1 -> (Luis,Tablet), (Ana,Auriculares), (Carlos,Impresora), (Marta,Webcam)
📦 Partición 2 -> (Luis,Ratón), (Carlos,SSD), (Ana,USB), (Marta,Silla)


compras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[136] at parallelize at cmd95.sc:1

In [96]:
val comprasPorCliente = compras.combineByKey(
    (productos: String) => List(productos),
    (productos: List[String], producto: String) => productos :+ producto,
    (p1: List[String], p2: List[String]) => p1 ++ p2
)

comprasPorCliente.collect().sortBy(_._1).foreach{ case (cliente, productos) =>
    println(s"$cliente: $productos")
}

Ana: List(Laptop, Mouse, Auriculares, USB)
Carlos: List(Impresora, SSD)
Luis: List(Teclado, Tablet, Ratón)
Marta: List(Monitor, Webcam, Silla)


comprasPorCliente: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[138] at combineByKey at cmd96.sc:4

## 🎯 Objetivo

Agrupar los productos comprados por cliente.

## ✅ Salida esperada (aproximada)

```
(Ana,List(Laptop, Mouse, Auriculares, USB))
(Luis,List(Teclado, Tablet, Ratón))
(Marta,List(Monitor, Webcam, Silla))
(Carlos,List(Impresora, SSD))
```

---

# 🧪 Ejercicio 3: Notas por estudiante

## 📥 Datos

In [97]:
val notas = sc.parallelize(List(
  ("Carlos", 8.5),
  ("Marta", 7.0),
  ("Carlos", 9.0),
  ("Ana", 6.5),
  ("Marta", 8.0),
  ("Ana", 7.5),
  ("Luis", 5.5),
  ("Carlos", 10.0),
  ("Luis", 6.0),
  ("Marta", 9.0),
  ("Ana", 8.5),
  ("Luis", 7.0)
), 3)

verParticiones(notas)

📦 Partición 0 -> (Carlos,8.5), (Marta,7.0), (Carlos,9.0), (Ana,6.5)
📦 Partición 1 -> (Marta,8.0), (Ana,7.5), (Luis,5.5), (Carlos,10.0)
📦 Partición 2 -> (Luis,6.0), (Marta,9.0), (Ana,8.5), (Luis,7.0)


notas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[139] at parallelize at cmd97.sc:1

In [98]:
val notasPorEstudiante = notas.combineByKey(
    (nota: Double) => List(nota),
    (notas: List[Double], nota: Double) => notas :+ nota,
    (n1: List[Double], n2: List[Double]) => n1 ++ n2
)

notasPorEstudiante.collect().sortBy(_._1).foreach{ case (alumno, notas) =>
    println(s"$alumno: $notas")
}

Ana: List(6.5, 7.5, 8.5)
Carlos: List(8.5, 9.0, 10.0)
Luis: List(5.5, 6.0, 7.0)
Marta: List(7.0, 8.0, 9.0)


notasPorEstudiante: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[141] at combineByKey at cmd98.sc:4

## 🎯 Objetivo

Agrupar las notas por estudiante.

## ✅ Salida esperada (aproximada)

```
(Carlos,List(8.5, 9.0, 10.0))
(Marta,List(7.0, 8.0, 9.0))
(Ana,List(6.5, 7.5, 8.5))
(Luis,List(5.5, 6.0, 7.0))
```

---

# 🧪 Ejercicio 4: Palabras por inicial

## 📥 Datos

In [99]:
val palabras = sc.parallelize(List(
  ("A", "Azure"),
  ("S", "Spark"),
  ("A", "Almond"),
  ("S", "Scala"),
  ("P", "Python"),
  ("J", "Java"),
  ("A", "Airflow"),
  ("S", "SQL"),
  ("P", "PostgreSQL"),
  ("J", "Jupyter"),
  ("A", "AWS"),
  ("P", "PySpark")
), 3)

verParticiones(palabras)

📦 Partición 0 -> (A,Azure), (S,Spark), (A,Almond), (S,Scala)
📦 Partición 1 -> (P,Python), (J,Java), (A,Airflow), (S,SQL)
📦 Partición 2 -> (P,PostgreSQL), (J,Jupyter), (A,AWS), (P,PySpark)


palabras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[142] at parallelize at cmd99.sc:1

In [100]:
val palabrasPorInicial = palabras.combineByKey(
    (palabra: String) => List(palabra),
    (palabras: List[String], palabra: String) => palabras :+ palabra,
    (p1: List[String], p2: List[String]) =>  p1 ++ p2 
)

palabrasPorInicial.collect().sortBy(_._1).foreach{ case (inicial, palabras) =>
    println(s"$inicial: $palabras")
}

A: List(Azure, Almond, Airflow, AWS)
J: List(Java, Jupyter)
P: List(Python, PostgreSQL, PySpark)
S: List(Spark, Scala, SQL)


palabrasPorInicial: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[144] at combineByKey at cmd100.sc:4

## 🎯 Objetivo

Agrupar palabras según su inicial.

## ✅ Salida esperada (aproximada)

```
(A,List(Azure, Almond, Airflow, AWS))
(S,List(Spark, Scala, SQL))
(P,List(Python, PostgreSQL, PySpark))
(J,List(Java, Jupyter))
```

---

# 🧠 Plantilla para resolver todos los ejercicios

```
val resultado = rdd.combineByKey(
  (v: TipoValor) => List(v),
  (acc: List[TipoValor], v: TipoValor) => acc :+ v,
  (acc1: List[TipoValor], acc2: List[TipoValor]) => acc1 ++ acc2
)

resultado.collect().foreach(println)
```

---

# 🧪 Caso de Estudio: Análisis de Actividad de Usuarios en una Plataforma Digital

---

## 🏢 Contexto empresarial

Una empresa de e-learning (tipo plataforma de cursos online) quiere analizar el comportamiento de sus usuarios. Cada vez que un usuario realiza una acción en la plataforma (ver un vídeo, completar una lección, hacer un test, etc.), se genera un registro. El equipo de Data Engineering necesita preparar los datos para que el equipo de Data Science pueda analizarlos posteriormente.

---

## 📥 Datos de entrada

Cada registro tiene la siguiente estructura:

```
usuario, tipo_evento, duracion_minutos
```

Ejemplo de dataset:

In [101]:
val eventos = sc.parallelize(List(
  ("user1", "video", 10.0),
  ("user2", "quiz", 5.0),
  ("user1", "video", 15.0),
  ("user3", "video", 20.0),
  ("user2", "video", 8.0),
  ("user1", "quiz", 7.0),
  ("user3", "quiz", 6.0),
  ("user2", "video", 12.0),
  ("user1", "video", 9.0),
  ("user3", "video", 11.0),
  ("user2", "quiz", 4.0),
  ("user3", "video", 13.0)
), 3)

eventos: org.apache.spark.rdd.RDD[(String, String, Double)] = ParallelCollectionRDD[145] at parallelize at cmd101.sc:1

---

## 🔍 Paso 1: Visualizar particiones

In [102]:
verParticiones(eventos)

📦 Partición 0 -> (user1,video,10.0), (user2,quiz,5.0), (user1,video,15.0), (user3,video,20.0)
📦 Partición 1 -> (user2,video,8.0), (user1,quiz,7.0), (user3,quiz,6.0), (user2,video,12.0)
📦 Partición 2 -> (user1,video,9.0), (user3,video,11.0), (user2,quiz,4.0), (user3,video,13.0)


---

Construir una estructura donde:

```
usuario -> List(duraciones)
```

Es decir, agrupar todas las duraciones de actividad por usuario.

## 🧠 ¿Por qué usar `combineByKey`?

Porque queremos:

- Crear una estructura personalizada (List[Double])
- Trabajar eficientemente en entorno distribuido
- Evitar el uso de `groupByKey` (menos eficiente)

---

## 🔄 Transformación necesaria

Primero debemos transformar los datos a:

```(usuario, duracion)```

---

## 🧪 Tarea propuesta

### 1️⃣ Transformar el dataset

In [103]:
val eventosPorUsuario = eventos.map { case (user, tipo, duracion) =>
  (user, duracion)
}

eventosPorUsuario: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[147] at map at cmd103.sc:1

---

### 2️⃣ Aplicar `combineByKey`

Debes implementar:

In [104]:
val resultado = eventosPorUsuario.combineByKey(
  (duracion: Double) => List(duracion),
  (duraciones: List[Double], duracion: Double) => duraciones :+ duracion,
  (d1: List[Double], d2: List[Double]) => d1 ++ d2 
)

resultado.collect().sortBy(_._1).foreach{ case (user, duraciones) =>
  println(s"$user: $duraciones")
}

user1: List(10.0, 15.0, 7.0, 9.0)
user2: List(5.0, 8.0, 12.0, 4.0)
user3: List(20.0, 6.0, 11.0, 13.0)


resultado: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[148] at combineByKey at cmd104.sc:4

---

## 📌 Resultado esperado (aproximado)

```
(user1,List(10.0, 15.0, 7.0, 9.0))
(user2,List(5.0, 8.0, 12.0, 4.0))
(user3,List(20.0, 6.0, 11.0, 13.0))
```

---

## 🚀 Extensión

Una vez tengas la lista de duraciones por usuario, calcula:

### 🔹 Tiempo total por usuario

```user -> suma_duraciones```

In [105]:
val sumaDuraciones = eventosPorUsuario.aggregateByKey(
    (0.0, 0)
)(
    (suma, duraciones) => (suma._1 + duraciones, suma._2 + 1),
    (suma1, suma2)   => (suma1._1 + suma2._1, suma1._2 + suma2._2)
)

sumaDuraciones.collect().sortBy(_._1).foreach{case(user, (suma,cuenta)) => 
    println(s"$user: $suma € - $cuenta eventos")
}

user1: 41.0 € - 4 eventos
user2: 29.0 € - 4 eventos
user3: 50.0 € - 4 eventos


sumaDuraciones: org.apache.spark.rdd.RDD[(String, (Double, Int))] = ShuffledRDD[149] at aggregateByKey at cmd105.sc:3

### 🔹 Tiempo medio por usuario

```user -> promedio_duraciones```

In [106]:
sumaDuraciones.collect().sortBy(_._1).foreach{case(user, (suma,cuenta)) => 
    val media = suma / cuenta
    println(s"$user: $suma € - $cuenta eventos - media: $media")
}
 

user1: 41.0 € - 4 eventos - media: 10.25
user2: 29.0 € - 4 eventos - media: 7.25
user3: 50.0 € - 4 eventos - media: 12.5


---

## 💡 Preguntas para reflexión

1. ¿Qué ocurre dentro de cada partición?
2. ¿Cuándo se ejecuta `mergeCombiners`?
3. ¿Por qué `combineByKey` es más flexible que `reduceByKey`?
4. ¿Qué pasaría si usas `groupByKey` en este caso?



---

# Sesión 2   - Spark DataFrames I: Introducción

---

# 🧠 Teoría

## 1. Limitaciones de los RDDs: por qué surgieron los DataFrames

En clases anteriores hemos usado RDDs para procesar datos. Los RDDs son potentes y flexibles, pero tienen un problema importante: **Spark no sabe nada sobre los datos que contienen**. Cuando escribes `rdd.map(linea => linea.split(","))`, Spark solo ve que tiene un RDD de cadenas y que le aplicas una función. No sabe qué hay dentro de esas cadenas, si son números o texto, si algunas columnas son nulas, ni cómo está estructurado el dato. Esto impide cualquier tipo de optimización automática.

| Problema en RDDs | Consecuencia |
| --- | --- |
| Sin conocimiento de la estructura | Spark no puede optimizar el plan de ejecución |
| Sin tipos de columna | No detecta errores hasta el momento de ejecutar |
| Sin nombres de campo | El código usa índices numéricos: `campos(0)`, `campos(2)` |
| Sin estadísticas de columna | No puede elegir el join más eficiente |

Los **DataFrames** nacen en Spark 1.3 para resolver exactamente estos problemas. Son la evolución natural de los RDDs para trabajo con datos estructurados.

> 💡 **Analogía:** un RDD es como una caja de cartón llena de papeles: sabes que hay papeles, pero no qué pone en ellos. Un DataFrame es como una hoja de cálculo con cabecera: cada columna tiene nombre, tipo y Spark puede consultarla, filtrarla y optimizarla sin que tú lo indiques.
> 

---

## 2. ¿Qué es un DataFrame en Spark?

Un **DataFrame** es una colección distribuida de datos organizada en **columnas con nombre y tipo definido**, similar conceptualmente a una tabla de base de datos relacional o a una hoja de cálculo.

```
+------+---------+------+----------+
|  id  | nombre  | edad | ciudad   |
+------+---------+------+----------+
|  1   | Ana     |  28  | Madrid   |
|  2   | Luis    |  34  | Barcelona|
|  3   | Marta   |  22  | Sevilla  |
+------+---------+------+----------+
```

Características clave:

- **Distribuido:** los datos están repartidos entre los nodos del clúster, igual que un RDD.
- **Inmutable:** cada transformación produce un nuevo DataFrame; el original no cambia.
- **Lazy:** las transformaciones no se ejecutan hasta que se llama a una acción.
- **Con schema:** cada columna tiene un nombre y un tipo de dato conocido por Spark.
- **Optimizado:** el motor Catalyst de Spark optimiza automáticamente el plan de ejecución.

### ¿Cómo se relaciona con los RDDs?

Un DataFrame es, internamente, un `RDD[Row]` con un schema adjunto. La diferencia es que Spark sí puede inspeccionar ese schema y generar código optimizado a partir de él.

```
DataFrame  =  RDD[Row]  +  Schema
                              ↓
                    Spark puede optimizar
```

---

## 3. `SparkSession`: el punto de entrada unificado

Para trabajar con DataFrames necesitamos una `SparkSession`. Es el objeto principal desde el que creamos DataFrames, ejecutamos SQL y accedemos a la configuración de Spark. En versiones anteriores de Spark existían `SparkContext`, `SQLContext` y `HiveContext` por separado. A partir de Spark 2.0, `SparkSession` los unifica a todos.

In [2]:


import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-core:4.1.1`
import org.apache.spark.sql.SparkSession
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.WARN)
Logger.getLogger("akka").setLevel(Level.WARN)

val spark = SparkSession.builder()
  .appName("DataFrames_Dia14")   // nombre visible en la Spark UI
  .master("local[*]")            // modo local, todos los cores disponibles
  .getOrCreate()                 // crea o reutiliza una sesión existente

// Desde SparkSession podemos acceder al SparkContext si lo necesitamos
val sc = spark.sparkContext

println(s"Spark ${spark.version} listo")

Spark 4.1.1 listo


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.log4j.{Level, Logger}
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@64916684
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@5b3c17ac

> 💡 `.getOrCreate()` es importante: si ya existe una `SparkSession` activa en el proceso (por ejemplo, en la celda anterior del notebook), la reutiliza en lugar de crear una nueva. Esto evita errores por sesiones duplicadas.
> 

### Accesos frecuentes desde `spark`

| Expresión | Qué devuelve |
| --- | --- |
| `spark.sparkContext` | El `SparkContext` subyacente (para RDDs) |
| `spark.version` | Versión de Spark en uso |
| `spark.read` | `DataFrameReader` para cargar datos |
| `spark.sql("SELECT ...")` | Ejecutar SQL directamente |
| `spark.catalog` | Acceso a tablas y vistas registradas |

---

## 4. Crear DataFrames

Hay varias formas de crear un DataFrame según el origen de los datos.

### 4.1 Desde una colección en memoria

La más sencilla para pruebas y aprendizaje. Usamos `createDataFrame` con una secuencia de tuplas y le damos nombres a las columnas con `.toDF(...)`.

In [3]:
import spark.implicits._
// Forma 1: Seq de tuplas + toDF con nombres de columna
val dfEmpleados = Seq(
  (1, "Ana García",    28, "Ingeniería"),
  (2, "Luis Martínez", 34, "Marketing"),
  (3, "Marta López",   22, "Ingeniería"),
  (4, "Pedro Ruiz",    41, "Dirección")
).toDF("id", "nombre", "edad", "departamento")

dfEmpleados.show()
// +---+-------------+----+------------+
// | id|       nombre|edad|departamento|
// +---+-------------+----+------------+
// |  1|   Ana García|  28|  Ingeniería|
// |  2|Luis Martínez|  34|   Marketing|
// |  3|  Marta López|  22|  Ingeniería|
// |  4|   Pedro Ruiz|  41|   Dirección|
// +---+-------------+----+------------+

+---+-------------+----+------------+
| id|       nombre|edad|departamento|
+---+-------------+----+------------+
|  1|   Ana García|  28|  Ingeniería|
|  2|Luis Martínez|  34|   Marketing|
|  3|  Marta López|  22|  Ingeniería|
|  4|   Pedro Ruiz|  41|   Dirección|
+---+-------------+----+------------+



import spark.implicits._
dfEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 2 more fields]

> ⚠️ **Nota importante para el entorno Almond:** para usar `.toDF(...)` necesitas importar las implicits de Spark. Añade esta línea después de crear la sesión:
> 
> 
> ```scala
> import spark.implicits._
> ```
> 
> Sin este import, el compilador no encontrará el método `.toDF`.
> 

## Pasos para crear un .csv y un .json en vscode desde el notebook

<aside>
💡

Pasos para crear datos sintéticos en csv desde VsCode

✅ 1. Crear CSV sintético en esa ruta:

In [4]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

// val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
val ruta = "./datos"
Files.createDirectories(Paths.get(ruta))

val contenidoCSV =
"""id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""

val pathCSV = s"$ruta/ventas.csv"

Files.write(
  Paths.get(pathCSV),
  contenidoCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")

CSV creado en: ./datos/ventas.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "./datos"
res4_3: java.nio.file.Path = ./datos
contenidoCSV: String = """id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""
pathCSV: String = "./datos/ventas.csv"
res4_6: java.nio.file.Path = ./datos/ventas.csv

✅ 2. Leer el CSV con Spark

In [5]:
val dfVentas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("./datos/ventas.csv")
  // .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv")

dfVentas.show(5)
dfVentas.printSchema()

+--------+----------+--------+-----------+--------+---------------+---------+
|id_venta|     fecha|producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+--------+-----------+--------+---------------+---------+
|       1|2026-04-01|Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|   Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02| Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02| Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|   Silla|    Oficina|       4|          120.0|Barcelona|
+--------+----------+--------+-----------+--------+---------------+---------+
only showing top 5 rows
root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- ciudad: string (nul

dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

</aside>

<aside>
💡

Pasos para crear un JSON simple:

1. En Spark, el JSON simple suele ser **JSON Lines**: un objeto JSON por línea.
    


In [6]:

    import java.nio.file.{Files, Paths}
    import java.nio.charset.StandardCharsets
    
    // val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos" // Usa tu propia ruta, sino te dará error
    val ruta = "./datos" // Usa tu propia ruta, sino te dará error
    Files.createDirectories(Paths.get(ruta))
    
    val jsonSimple =
    """{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
    {"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
    {"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
    {"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""
    
    val rutaJsonSimple = s"$ruta/empleados_simple.json"
    
    Files.write(
      Paths.get(rutaJsonSimple),
      jsonSimple.getBytes(StandardCharsets.UTF_8)
    )
    
    println(s"JSON simple creado en: $rutaJsonSimple")


JSON simple creado en: ./datos/empleados_simple.json


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "./datos"
res6_3: java.nio.file.Path = ./datos
jsonSimple: String = """{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
    {"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
    {"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
    {"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""
rutaJsonSimple: String = "./datos/empleados_simple.json"
res6_6: java.nio.file.Path = ./datos/empleados_simple.json


    
2. Leerlo con Spark:

In [7]:
val dfJsonSimple = spark.read
  .option("inferSchema", "true")
  .json(rutaJsonSimple)

dfJsonSimple.show(false)
dfJsonSimple.printSchema()

+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonSimple: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

</aside>

<aside>
💡

Pasos para crear un JSON multilínea:

1. Este formato tiene normalmente un **array JSON** con varios objetos dentro.
    


In [8]:

    val jsonMultilinea =
    """[
      {
        "id": 1,
        "nombre": "Ana García",
        "edad": 28,
        "departamento": "Ingeniería"
      },
      {
        "id": 2,
        "nombre": "Luis Martínez",
        "edad": 34,
        "departamento": "Marketing"
      },
      {
        "id": 3,
        "nombre": "Marta López",
        "edad": 22,
        "departamento": "Ingeniería"
      },
      {
        "id": 4,
        "nombre": "Pedro Ruiz",
        "edad": 41,
        "departamento": "Dirección"
      }
    ]"""
    
    val rutaJsonMultilinea = s"$ruta/empleados_multilinea.json"
    
    Files.write(
      Paths.get(rutaJsonMultilinea),
      jsonMultilinea.getBytes(StandardCharsets.UTF_8)
    )
    
    println(s"JSON multilínea creado en: $rutaJsonMultilinea")


JSON multilínea creado en: ./datos/empleados_multilinea.json


jsonMultilinea: String = """[
      {
        "id": 1,
        "nombre": "Ana García",
        "edad": 28,
        "departamento": "Ingeniería"
      },
      {
        "id": 2,
        "nombre": "Luis Martínez",
        "edad": 34,
        "departamento": "Marketing"
      },
      {
        "id": 3,
        "nombre": "Marta López",
        "edad": 22,
        "departamento": "Ingeniería"
      },
      {
        "id": 4,
        "nombre": "Pedro Ruiz",
        "edad": 41,
        "departamento": "Dirección"
      }
    ]"""
rutaJsonMultilinea: String = "./datos/empleados_multilinea.json"
res8_2: java.nio.file.Path = ./datos/empleados_multilinea.json

    
2. Leerlo con Spark:
    


In [9]:
    val dfJsonMultilinea = spark.read
      .option("multiline", "true")
      .option("inferSchema", "true")
      .json(rutaJsonMultilinea)
    
    dfJsonMultilinea.show(false)
    dfJsonMultilinea.printSchema()


+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonMultilinea: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

  
3. La diferencia clave es esta opción:
    
    ```scala
    .option("multiline", "true")
    ```
    
</aside>



### 4.2 Desde un fichero CSV

In [10]:
val dfVentas = spark.read
  .option("header", "true")       // primera fila como nombres de columna
  .option("inferSchema", "true")  // detectar tipos automáticamente
  .csv("./datos/ventas.csv")
  // .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv")

dfVentas.show(5)         // muestra las primeras 5 filas


+--------+----------+--------+-----------+--------+---------------+---------+
|id_venta|     fecha|producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+--------+-----------+--------+---------------+---------+
|       1|2026-04-01|Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|   Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02| Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02| Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|   Silla|    Oficina|       4|          120.0|Barcelona|
+--------+----------+--------+-----------+--------+---------------+---------+
only showing top 5 rows


dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

In [11]:
dfVentas.printSchema()   // muestra la estructura con tipos

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- ciudad: string (nullable = true)



### 4.3 Desde un fichero JSON

In [12]:
val dfClientes = spark.read
  .option("multiline", "true")    // para JSON con objetos multilínea
  .json("./datos/empleados_multilinea.json")
  // .json("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json")

dfClientes.show()

+------------+----+---+-------------+
|departamento|edad| id|       nombre|
+------------+----+---+-------------+
|  Ingeniería|  28|  1|   Ana García|
|   Marketing|  34|  2|Luis Martínez|
|  Ingeniería|  22|  3|  Marta López|
|   Dirección|  41|  4|   Pedro Ruiz|
+------------+----+---+-------------+



dfClientes: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [13]:
dfClientes.printSchema()

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



### 4.4 Desde un fichero Parquet

In [13]:
// // Parquet lleva el schema integrado en el propio fichero
// val dfParquet = spark.read
//   .parquet("./datos/pedidos.parquet")
//   // .parquet("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.parquet")

// dfParquet.show()

> 💡Empezaremos con CSV y JSON. El formato Parquet lo veremos en profundidad en la clase 17.
> 

---

## 5. Schema: la estructura del DataFrame

El **schema** describe las columnas del DataFrame: sus nombres, tipos de dato y si admiten valores nulos. Es la información que hace que Spark pueda optimizar las operaciones.

### 5.1 Inspeccionar el schema con `printSchema()`

In [14]:
dfEmpleados.printSchema()
// root
//  |-- id: integer (nullable = true)
//  |-- nombre: string (nullable = true)
//  |-- edad: integer (nullable = true)
//  |-- departamento: string (nullable = true)

root
 |-- id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = false)
 |-- departamento: string (nullable = true)



La salida muestra un árbol con:

- **Nombre** de cada columna
- **Tipo de dato** (`integer`, `string`, `double`, `boolean`, `date`, ...)
- **nullable:** si la columna puede contener valores nulos (`true` = sí puede)

### 5.2 Schema por inferencia vs. schema manual

Cuando usamos `inferSchema = true`, Spark lee una muestra del fichero y adivina los tipos. Es cómodo pero tiene dos inconvenientes: es más lento (tiene que leer los datos dos veces) y puede equivocarse con columnas ambiguas (por ejemplo, un código postal "01234" podría inferirse como `integer` y perder el cero inicial).

La alternativa es definir el schema **manualmente** con `StructType` y `StructField`:

In [15]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

import org.apache.spark.sql.types._

// val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
val ruta = "./datos"
Files.createDirectories(Paths.get(ruta))

val contenidoPedidos =
"""id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""

val rutaPedidos = s"$ruta/pedidos.csv"

Files.write(
  Paths.get(rutaPedidos),
  contenidoPedidos.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado correctamente en: $rutaPedidos")

CSV creado correctamente en: ./datos/pedidos.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import org.apache.spark.sql.types._
ruta: String = "./datos"
res15_4: java.nio.file.Path = ./datos
contenidoPedidos: String = """id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""
rutaPedidos: String = "./datos/pedidos.csv"
res15_7: java.nio.file.Path = ./datos/pedidos.csv

In [16]:
import org.apache.spark.sql.types._
val schemaPedidos = StructType(List(
  StructField("id_pedido", IntegerType, nullable = false),
  StructField("id_cliente", IntegerType, nullable = false),
  StructField("fecha", StringType, nullable = true),
  StructField("importe", DoubleType, nullable = true),
  StructField("completado", BooleanType, nullable = true)
))

val dfPedidos = spark.read
  .option("header", "true")
  .schema(schemaPedidos)
  .csv("./datos/pedidos.csv")
  // .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv")

dfPedidos.show(false)


+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|fecha     |importe|completado|
+---------+----------+----------+-------+----------+
|1        |101       |2026-04-01|250.75 |true      |
|2        |102       |2026-04-01|99.9   |false     |
|3        |103       |2026-04-02|430.0  |true      |
|4        |101       |2026-04-03|120.5  |true      |
|5        |104       |2026-04-04|75.25  |false     |
|6        |105       |2026-04-05|680.4  |true      |
+---------+----------+----------+-------+----------+



import org.apache.spark.sql.types._
schemaPedidos: StructType = Seq(
  StructField(
    name = "id_pedido",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "id_cliente",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "fecha",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "importe",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "completado",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfPedidos: org.apache.spark.sql.package.DataFrame = [id_pedido: int, id_cliente: int ... 3 more fields]

In [17]:
dfPedidos.printSchema()

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



Salida esperada:

```
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)
 ```

In [18]:
import org.apache.spark.sql.types._

val schemaPedidos = StructType(List(
  StructField("id_pedido",   IntegerType, nullable = false),
  StructField("id_cliente",  IntegerType, nullable = false),
  StructField("fecha",       StringType,  nullable = true),
  StructField("importe",     DoubleType,  nullable = true),
  StructField("completado",  BooleanType, nullable = true)
))

val dfPedidos = spark.read
  .option("header", "true")
  .schema(schemaPedidos)          // usamos el schema manual
  .csv("./datos/pedidos.csv")
  // .csv("C:/Curso-Scala/datos/pedidos.csv")

dfPedidos.printSchema()
// root
//  |-- id_pedido: integer (nullable = false)
//  |-- id_cliente: integer (nullable = false)
//  |-- fecha: string (nullable = true)
//  |-- importe: double (nullable = true)
//  |-- completado: boolean (nullable = true)

val df = dfPedidos

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



import org.apache.spark.sql.types._
schemaPedidos: StructType = Seq(
  StructField(
    name = "id_pedido",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "id_cliente",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "fecha",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "importe",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "completado",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfPedidos: org.apache.spark.sql.package.DataFrame = [id_pedido: int, id_cliente: int ... 3 more fields]
df: org.apache.spark.sql.package.DataFrame = [id_pedido: int, id_cliente: int ... 3 more fields]

### Tipos de dato más habituales en Spark

| Tipo Spark | Clase Scala equivalente | Ejemplo de valor |
| --- | --- | --- |
| `IntegerType` | `Int` | `42` |
| `LongType` | `Long` | `1234567890L` |
| `DoubleType` | `Double` | `3.14` |
| `StringType` | `String` | `"hola"` |
| `BooleanType` | `Boolean` | `true` |
| `DateType` | `java.sql.Date` | `"2024-03-15"` |
| `TimestampType` | `java.sql.Timestamp` | `"2024-03-15 10:30:00"` |

---

## 6. Explorar un DataFrame recién cargado

Antes de transformar datos, siempre conviene explorar el DataFrame con estas operaciones básicas:

### `show(n)` — ver filas

In [19]:
df.show()        // primeras 20 filas (por defecto)


+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|     fecha|importe|completado|
+---------+----------+----------+-------+----------+
|        1|       101|2026-04-01| 250.75|      true|
|        2|       102|2026-04-01|   99.9|     false|
|        3|       103|2026-04-02|  430.0|      true|
|        4|       101|2026-04-03|  120.5|      true|
|        5|       104|2026-04-04|  75.25|     false|
|        6|       105|2026-04-05|  680.4|      true|
+---------+----------+----------+-------+----------+



In [20]:
df.show(5)       // primeras 5 filas

+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|     fecha|importe|completado|
+---------+----------+----------+-------+----------+
|        1|       101|2026-04-01| 250.75|      true|
|        2|       102|2026-04-01|   99.9|     false|
|        3|       103|2026-04-02|  430.0|      true|
|        4|       101|2026-04-03|  120.5|      true|
|        5|       104|2026-04-04|  75.25|     false|
+---------+----------+----------+-------+----------+
only showing top 5 rows


In [21]:
df.show(5, truncate = false)  // sin truncar textos largos

+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|fecha     |importe|completado|
+---------+----------+----------+-------+----------+
|1        |101       |2026-04-01|250.75 |true      |
|2        |102       |2026-04-01|99.9   |false     |
|3        |103       |2026-04-02|430.0  |true      |
|4        |101       |2026-04-03|120.5  |true      |
|5        |104       |2026-04-04|75.25  |false     |
+---------+----------+----------+-------+----------+
only showing top 5 rows


### `printSchema()` — ver estructura

In [22]:
df.printSchema()
// root
//  |-- nombre: string (nullable = true)
//  |-- edad: integer (nullable = true)

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



### `count()` — número de filas

In [23]:
val total = df.count()
println(s"Total de registros: $total")

Total de registros: 6


total: Long = 6L

### `columns` — nombres de columnas

In [30]:
df.columns.foreach(println)
// id
// nombre
// edad
// departamento

id_pedido
id_cliente
fecha
importe
completado


### `describe()` — estadísticas descriptivas

Calcula automáticamente count, media, desviación estándar, mínimo y máximo para las columnas numéricas:

In [29]:
df.describe().show()
// +-------+------------------+------+
// |summary|              edad|    id|
// +-------+------------------+------+
// |  count|                 4|     4|
// |   mean|             31.25|   2.5|
// | stddev|  8.26...         |  1.29|
// |    min|                22|     1|
// |    max|                41|     4|
// +-------+------------------+------+


+-------+------------------+------------------+----------+-----------------+
|summary|         id_pedido|        id_cliente|     fecha|          importe|
+-------+------------------+------------------+----------+-----------------+
|  count|                 6|                 6|         6|                6|
|   mean|               3.5|102.66666666666667|      NULL|276.1333333333333|
| stddev|1.8708286933869707|1.6329931618554512|      NULL|238.0692497292892|
|    min|                 1|               101|2026-04-01|            75.25|
|    max|                 6|               105|2026-04-05|            680.4|
+-------+------------------+------------------+----------+-----------------+



In [37]:
// También puedes limitar a columnas concretas:
// df.describe("edad", "id").show()

### `dtypes` — tipos de columnas como array

In [32]:
df.dtypes.foreach { case (nombre, tipo) =>
  println(s"$nombre → $tipo")
}
// id → IntegerType
// nombre → StringType
// edad → IntegerType
// departamento → StringType

id_pedido → IntegerType
id_cliente → IntegerType
fecha → StringType
importe → DoubleType
completado → BooleanType


---

## 📊 Resumen de la sesión

| Concepto | Qué es | Cómo se usa |
| --- | --- | --- |
| `DataFrame` | Tabla distribuida con schema | Principal estructura de Spark para datos estructurados |
| `SparkSession` | Punto de entrada unificado | `SparkSession.builder().appName(...).master(...).getOrCreate()` |
| `spark.read.csv(...)` | Carga desde CSV | Con opciones `header`, `inferSchema`, `schema` |
| `spark.read.json(...)` | Carga desde JSON | Con opción `multiline` si hace falta |
| `StructType` / `StructField` | Schema manual | Más robusto y rápido que `inferSchema` |
| `show()` | Ver filas | `show()`, `show(n)`, `show(n, false)` |
| `printSchema()` | Ver estructura | Árbol con nombres, tipos y nullable |
| `describe()` | Estadísticas | count, mean, stddev, min, max por columna |

---

# 💻 Práctica

---

## 🗂️ Paso previo: crear los ficheros de datos

Antes de empezar con el notebook, crea los ficheros de datos que usaremos. Abre el **Bloc de notas** o cualquier editor de texto como VsCode y guarda los siguientes ficheros en `C:\Curso-Scala\datos\` (crea la carpeta si no existe).

---

### Fichero 1 — `empleados.csv`

```
id,nombre,edad,departamento,salario,activo
1,Ana García,28,Ingeniería,42000,true
2,Luis Martínez,34,Marketing,38000,true
3,Marta López,22,Ingeniería,35000,true
4,Pedro Ruiz,41,Dirección,75000,true
5,Carmen Díaz,29,Marketing,36500,true
6,Jorge Santos,38,Ingeniería,48000,false
7,Elena Vega,31,RRHH,33000,true
8,Tomás Gil,45,Dirección,82000,true
9,Laura Prieto,26,Ingeniería,39000,true
10,Andrés Mora,33,Marketing,41000,false
```

---

### Fichero 2 — `productos.json`

```json
[
  {"id": 101, "nombre": "Laptop Pro", "categoria": "Informatica", "precio": 1299.99, "stock": 45},
  {"id": 102, "nombre": "Teclado Inalámbrico", "categoria": "Perifericos", "precio": 59.90, "stock": 120},
  {"id": 103, "nombre": "Monitor 27\"", "categoria": "Monitores", "precio": 349.00, "stock": 30},
  {"id": 104, "nombre": "Ratón Óptico", "categoria": "Perifericos", "precio": 24.95, "stock": 200},
  {"id": 105, "nombre": "Auriculares USB", "categoria": "Audio", "precio": 89.50, "stock": 75},
  {"id": 106, "nombre": "Webcam HD", "categoria": "Perifericos", "precio": 79.00, "stock": 60},
  {"id": 107, "nombre": "Disco SSD 1TB", "categoria": "Almacenamiento", "precio": 119.99, "stock": 90},
  {"id": 108, "nombre": "Hub USB-C", "categoria": "Perifericos", "precio": 44.90, "stock": 150}
]
```

---

## 🔧 Celda de inicialización

In [33]:
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.WARN)
Logger.getLogger("akka").setLevel(Level.WARN)

import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("Dia14S1_DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._   // necesario para .toDF() y otras conversiones

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo")

26/04/28 14:36:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ Spark 4.1.1 listo


import org.apache.log4j.{Level, Logger}
import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@64916684
import spark.implicits._

---

## 🔹 Ejercicio 1 — DataFrame desde colección en memoria

Crea tu primer DataFrame desde datos en el propio notebook y practica los métodos de exploración.

In [34]:
// Crear DataFrame desde una secuencia de tuplas
val dfCursos = Seq(
  (1, "Big Data con Scala", "Avanzado",  140, 4.8),
  (2, "Python para Datos",  "Intermedio", 80, 4.6),
  (3, "SQL Empresarial",    "Básico",     40, 4.9),
  (4, "Machine Learning",   "Avanzado",  120, 4.7),
  (5, "Power BI",           "Básico",     60, 4.5)
).toDF("id", "titulo", "nivel", "horas", "valoracion")

// Ver las filas
println("=== Contenido del DataFrame ===")
dfCursos.show()

// Ver la estructura
println("=== Schema del DataFrame ===")
dfCursos.printSchema()

// Número de filas y columnas
println(s"Filas: ${dfCursos.count()}")
println(s"Columnas: ${dfCursos.columns.length}")
println(s"Nombres de columnas: ${dfCursos.columns.mkString(", ")}")

=== Contenido del DataFrame ===
+---+------------------+----------+-----+----------+
| id|            titulo|     nivel|horas|valoracion|
+---+------------------+----------+-----+----------+
|  1|Big Data con Scala|  Avanzado|  140|       4.8|
|  2| Python para Datos|Intermedio|   80|       4.6|
|  3|   SQL Empresarial|    Básico|   40|       4.9|
|  4|  Machine Learning|  Avanzado|  120|       4.7|
|  5|          Power BI|    Básico|   60|       4.5|
+---+------------------+----------+-----+----------+

=== Schema del DataFrame ===
root
 |-- id: integer (nullable = false)
 |-- titulo: string (nullable = true)
 |-- nivel: string (nullable = true)
 |-- horas: integer (nullable = false)
 |-- valoracion: double (nullable = false)

Filas: 5
Columnas: 5
Nombres de columnas: id, titulo, nivel, horas, valoracion


dfCursos: org.apache.spark.sql.package.DataFrame = [id: int, titulo: string ... 3 more fields]

**Salida esperada:**

```
=== Contenido del DataFrame ===
+---+------------------+-----------+-----+----------+
| id|            titulo|      nivel|horas|valoracion|
+---+------------------+-----------+-----+----------+
|  1|Big Data con Scala|   Avanzado|  140|       4.8|
|  2| Python para Datos|Intermedio |   80|       4.6|
|  3|   SQL Empresarial|     Básico|   40|       4.9|
|  4|  Machine Learning|   Avanzado|  120|       4.7|
|  5|          Power BI|     Básico|   60|       4.5|
+---+------------------+-----------+-----+----------+

=== Schema del DataFrame ===
root
 |-- id: integer (nullable = false)
 |-- titulo: string (nullable = true)
 |-- nivel: string (nullable = true)
 |-- horas: integer (nullable = false)
 |-- valoracion: double (nullable = false)

Filas: 5
Columnas: 5
Nombres de columnas: id, titulo, nivel, horas, valoracion
```

**Estadísticas:**

In [35]:
// Estadísticas descriptivas de columnas numéricas
println("=== Estadísticas descriptivas ===")
dfCursos.describe("horas", "valoracion").show()

=== Estadísticas descriptivas ===
+-------+-----------------+------------------+
|summary|            horas|        valoracion|
+-------+-----------------+------------------+
|  count|                5|                 5|
|   mean|             88.0|               4.7|
| stddev|41.47288270665544|0.1581138830084192|
|    min|               40|               4.5|
|    max|              140|               4.9|
+-------+-----------------+------------------+



**Salida esperada:**

```
=== Estadísticas descriptivas ===
+-------+------------------+------------------+
|summary|             horas|        valoracion|
+-------+------------------+------------------+
|  count|                 5|                 5|
|   mean|              88.0|              4.70|
| stddev|  38.47...        |  0.152...        |
|    min|                40|               4.5|
|    max|               140|               4.9|
+-------+------------------+------------------+
```

---

## 🔹 Ejercicio 2 — Cargar un CSV con inferencia de schema

**Carga con inferSchema:**

In [38]:
val dfEmpleados = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("./datos/empleados.csv")
  // .csv("C:/Curso-Scala/datos/empleados.csv")

println("=== Primeras filas ===")
dfEmpleados.show()

println("=== Schema inferido ===")
dfEmpleados.printSchema()

=== Primeras filas ===
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|  1|   Ana García|  28|  Ingeniería|  42000|  true|
|  2|Luis Martínez|  34|   Marketing|  38000|  true|
|  3|  Marta López|  22|  Ingeniería|  35000|  true|
|  4|   Pedro Ruiz|  41|   Dirección|  75000|  true|
|  5|  Carmen Díaz|  29|   Marketing|  36500|  true|
|  6| Jorge Santos|  38|  Ingeniería|  48000| false|
|  7|   Elena Vega|  31|        RRHH|  33000|  true|
|  8|    Tomás Gil|  45|   Dirección|  82000|  true|
|  9| Laura Prieto|  26|  Ingeniería|  39000|  true|
| 10|  Andrés Mora|  33|   Marketing|  41000| false|
+---+-------------+----+------------+-------+------+

=== Schema inferido ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: integer (nullable = true)
 |-- activo

dfEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 4 more fields]

**Salida esperada:**

```
=== Primeras filas ===
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|  1|   Ana García|  28|  Ingeniería|  42000|  true|
|  2|Luis Martínez|  34|   Marketing|  38000|  true|
...
+---+-------------+----+------------+-------+------+

=== Schema inferido ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: integer (nullable = true)
 |-- activo: boolean (nullable = true)
```

**schema manual para comparar:**

In [39]:
// Ahora definimos el schema manualmente
val schemaEmpleados = StructType(List(
  StructField("id",           IntegerType, nullable = false),
  StructField("nombre",       StringType,  nullable = true),
  StructField("edad",         IntegerType, nullable = true),
  StructField("departamento", StringType,  nullable = true),
  StructField("salario",      DoubleType,  nullable = true),  // Double en vez de Int
  StructField("activo",       BooleanType, nullable = true)
))

val dfEmpleadosTyped = spark.read
  .option("header", "true")
  .schema(schemaEmpleados)
  .csv("./datos/empleados.csv")
  // .csv("C:/Curso-Scala/datos/empleados.csv")

println("=== Schema manual aplicado ===")
dfEmpleadosTyped.printSchema()

// Diferencia: el campo salario ahora es DoubleType
// Útil si luego vamos a calcular medias o porcentajes

=== Schema manual aplicado ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: double (nullable = true)
 |-- activo: boolean (nullable = true)



schemaEmpleados: StructType = Seq(
  StructField(
    name = "id",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "nombre",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "edad",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "departamento",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "salario",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "activo",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfEmpleadosTyped: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 4 more fields]

**Exploración con dtypes:**

In [40]:
println("=== Tipos de cada columna ===")
dfEmpleadosTyped.dtypes.foreach { case (col, tipo) =>
  println(f"  $col%-15s → $tipo")
}

println(s"\nTotal empleados: ${dfEmpleadosTyped.count()}")

println("\n=== Estadísticas de edad y salario ===")
dfEmpleadosTyped.describe("edad", "salario").show()

=== Tipos de cada columna ===
  id              → IntegerType
  nombre          → StringType
  edad            → IntegerType
  departamento    → StringType
  salario         → DoubleType
  activo          → BooleanType

Total empleados: 10

=== Estadísticas de edad y salario ===
+-------+------------------+-----------------+
|summary|              edad|          salario|
+-------+------------------+-----------------+
|  count|                10|               10|
|   mean|              32.7|          46950.0|
| stddev|7.0561242115547325|17211.83378441188|
|    min|                22|          33000.0|
|    max|                45|          82000.0|
+-------+------------------+-----------------+



**Salida esperada:**

```
=== Tipos de cada columna ===
  id              → IntegerType
  nombre          → StringType
  edad            → IntegerType
  departamento    → StringType
  salario         → DoubleType
  activo          → BooleanType

Total empleados: 10

=== Estadísticas de edad y salario ===
+-------+------------------+------------------+
|summary|              edad|           salario|
+-------+------------------+------------------+
|  count|                10|                10|
|   mean|              32.7|           47050.0|
| stddev|   6.94...        |   17156.7...     |
|    min|                22|           33000.0|
|    max|                45|           82000.0|
+-------+------------------+------------------+
```

---

## 🔹 Ejercicio 3 — Cargar y explorar un fichero JSON

In [41]:
val dfProductos = spark.read
  .option("multiline", "true")
  .json("./datos/productos.json")
  // .json("C:/Curso-Scala/datos/productos.json")

println("=== Productos cargados ===")
dfProductos.show(truncate = false)

println("=== Schema del JSON ===")
dfProductos.printSchema()

=== Productos cargados ===
+--------------+---+-------------------+-------+-----+
|categoria     |id |nombre             |precio |stock|
+--------------+---+-------------------+-------+-----+
|Informatica   |101|Laptop Pro         |1299.99|45   |
|Perifericos   |102|Teclado Inalámbrico|59.9   |120  |
|Monitores     |103|Monitor 27"        |349.0  |30   |
|Perifericos   |104|Ratón Óptico       |24.95  |200  |
|Audio         |105|Auriculares USB    |89.5   |75   |
|Perifericos   |106|Webcam HD          |79.0   |60   |
|Almacenamiento|107|Disco SSD 1TB      |119.99 |90   |
|Perifericos   |108|Hub USB-C          |44.9   |150  |
+--------------+---+-------------------+-------+-----+

=== Schema del JSON ===
root
 |-- categoria: string (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: long (nullable = true)



dfProductos: org.apache.spark.sql.package.DataFrame = [categoria: string, id: bigint ... 3 more fields]

**Salida esperada:**

```
=== Productos cargados ===
+---+-------------------+--------------+-------+-----+
| id|             nombre|     categoria| precio|stock|
+---+-------------------+--------------+-------+-----+
|101|         Laptop Pro|   Informatica|1299.99|   45|
|102|Teclado Inalámbrico|   Perifericos|  59.90|  120|
|103|        Monitor 27"|     Monitores| 349.00|   30|
...
+---+-------------------+--------------+-------+-----+

=== Schema del JSON ===
root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: long (nullable = true)
```

> 💡 Fíjate en que Spark ha inferido `id` y `stock` como `LongType` (entero de 64 bits) en el JSON, mientras que en el CSV los infería como `IntegerType`. Esto es normal: el formato JSON no distingue entre `Int` y `Long`, y Spark elige el tipo más amplio por seguridad. En producción, un schema manual resuelve esta ambigüedad.
> 

**Exploración de columnas y estadísticas:**

In [42]:
println(s"Total de productos: ${dfProductos.count()}")

println("\n=== Categorías disponibles ===")
dfProductos.select("categoria").distinct().show()

println("=== Rango de precios ===")
dfProductos.describe("precio", "stock").show()

Total de productos: 8

=== Categorías disponibles ===
+--------------+
|     categoria|
+--------------+
|   Perifericos|
|   Informatica|
|     Monitores|
|Almacenamiento|
|         Audio|
+--------------+

=== Rango de precios ===
+-------+----------------+-----------------+
|summary|          precio|            stock|
+-------+----------------+-----------------+
|  count|               8|                8|
|   mean|       258.40375|            96.25|
| stddev|433.007817248389|57.36786058910885|
|    min|           24.95|               30|
|    max|         1299.99|              200|
+-------+----------------+-----------------+



**Salida esperada:**

```
Total de productos: 8

=== Categorías disponibles ===
+--------------+
|     categoria|
+--------------+
|   Informatica|
|   Perifericos|
|     Monitores|
|         Audio|
|Almacenamiento|
+--------------+

=== Rango de precios ===
+-------+------------------+------------------+
|summary|            precio|             stock|
+-------+------------------+------------------+
|  count|                 8|                 8|
|   mean|         258.40375|            96.25 |
| stddev|   411.0...       |    55.9...       |
|    min|             24.95|              30.0|
|    max|           1299.99|             200.0|
+-------+------------------+------------------+
```

---

## 🔹 Ejercicio 4 — Comparar inferencia vs. schema manual en JSON

In [43]:
// Schema manual: controlamos que id y stock sean Int, no Long
val schemaProductos = StructType(List(
  StructField("id",        IntegerType, nullable = false),
  StructField("nombre",    StringType,  nullable = true),
  StructField("categoria", StringType,  nullable = true),
  StructField("precio",    DoubleType,  nullable = true),
  StructField("stock",     IntegerType, nullable = true)
))

val dfProductosTyped = spark.read
  .option("multiline", "true")
  .schema(schemaProductos)
  .json("./datos/productos.json")
  // .json("C:/Curso-Scala/datos/productos.json")

println("=== Schema controlado ===")
dfProductosTyped.printSchema()

println("\n=== Comparación de tipos ===")
println("Con inferSchema:")
dfProductos.dtypes.foreach { case (c, t) => println(s"  $c → $t") }

println("\nCon schema manual:")
dfProductosTyped.dtypes.foreach { case (c, t) => println(s"  $c → $t") }

=== Schema controlado ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: integer (nullable = true)


=== Comparación de tipos ===
Con inferSchema:
  categoria → StringType
  id → LongType
  nombre → StringType
  precio → DoubleType
  stock → LongType

Con schema manual:
  id → IntegerType
  nombre → StringType
  categoria → StringType
  precio → DoubleType
  stock → IntegerType


schemaProductos: StructType = Seq(
  StructField(
    name = "id",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "nombre",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "categoria",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "precio",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "stock",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  )
)
dfProductosTyped: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

**Salida esperada:**

```
=== Schema controlado ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: integer (nullable = true)

=== Comparación de tipos ===
Con inferSchema:
  id → LongType
  nombre → StringType
  categoria → StringType
  precio → DoubleType
  stock → LongType

Con schema manual:
  id → IntegerType
  nombre → StringType
  categoria → StringType
  precio → DoubleType
  stock → IntegerType
```

---

## 🔹 Ejercicio 5 — `show` con opciones avanzadas

In [44]:
// show() tiene tres variantes útiles
println("show() por defecto — 20 filas, textos truncados a 20 chars:")
dfEmpleados.show()

println("show(3) — solo las primeras 3 filas:")
dfEmpleados.show(3)

println("show(3, truncate = false) — 3 filas, textos completos:")
dfEmpleados.show(3, truncate = false)

// columns devuelve un Array[String]
println(s"\nColumnas del DataFrame de empleados:")
println(dfEmpleados.columns.mkString(" | "))
// id | nombre | edad | departamento | salario | activo

show() por defecto — 20 filas, textos truncados a 20 chars:
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|  1|   Ana García|  28|  Ingeniería|  42000|  true|
|  2|Luis Martínez|  34|   Marketing|  38000|  true|
|  3|  Marta López|  22|  Ingeniería|  35000|  true|
|  4|   Pedro Ruiz|  41|   Dirección|  75000|  true|
|  5|  Carmen Díaz|  29|   Marketing|  36500|  true|
|  6| Jorge Santos|  38|  Ingeniería|  48000| false|
|  7|   Elena Vega|  31|        RRHH|  33000|  true|
|  8|    Tomás Gil|  45|   Dirección|  82000|  true|
|  9| Laura Prieto|  26|  Ingeniería|  39000|  true|
| 10|  Andrés Mora|  33|   Marketing|  41000| false|
+---+-------------+----+------------+-------+------+

show(3) — solo las primeras 3 filas:
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|

---

# 🏢 Caso de Estudio Propuesto : AgroData Cooperativa

---

## 🌾 Contexto del caso

**AgroData Cooperativa** es una organización agrícola que agrupa a productores de frutas y verduras de cinco provincias españolas. Hasta ahora, cada delegación provincial guardaba sus datos en hojas de cálculo independientes. El nuevo responsable de datos ha decidido centralizar toda la información en un sistema de análisis distribuido con Apache Spark.

El departamento de datos ha exportado la información existente en dos formatos:

- Un fichero **CSV** con el registro de todas las **parcelas** productivas de la cooperativa.
- Un fichero **JSON** con el catálogo de **productos** cultivados y sus características de mercado.

Tu tarea es ser el analista de datos que ingiere estos ficheros en Spark por primera vez, verifica que los datos han cargado correctamente, examina su estructura y extrae un primer informe de situación para la dirección.



---

## 📦 Datos del caso

Crea los siguientes ficheros en `C:\Curso-Scala\datos\agrodata\` antes de abrir el notebook.

---

### Fichero 1 — `parcelas.csv`



```
id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
P003,Almería,Níjar,25.0,Tomate,2012,true,85000
P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
P005,Murcia,Totana,15.2,Limón,2016,true,22000
P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
P010,Almería,Vícar,11.6,Pepino,2019,true,74000
P011,Murcia,Alhama,7.9,Limón,2017,true,20500
P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
P014,Almería,Roquetas,28.7,Tomate,2011,true,88000
P015,Murcia,Mazarrón,16.5,Pimiento,2018,true,58000
```



---

### Fichero 2 — `productos.json`



```json
[
  {
    "codigo": "NAR",
    "nombre": "Naranja",
    "familia": "Citrico",
    "precio_mercado_euro_kg": 0.45,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "FRE",
    "nombre": "Fresa",
    "familia": "Baya",
    "precio_mercado_euro_kg": 2.10,
    "demanda_exportacion": "Muy Alta",
    "certificacion_eco": true
  },
  {
    "codigo": "TOM",
    "nombre": "Tomate",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 0.85,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "ACE",
    "nombre": "Aceituna",
    "familia": "Oleaginosa",
    "precio_mercado_euro_kg": 0.60,
    "demanda_exportacion": "Media",
    "certificacion_eco": true
  },
  {
    "codigo": "LIM",
    "nombre": "Limón",
    "familia": "Citrico",
    "precio_mercado_euro_kg": 0.55,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "PIM",
    "nombre": "Pimiento",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 1.20,
    "demanda_exportacion": "Muy Alta",
    "certificacion_eco": true
  },
  {
    "codigo": "PEP",
    "nombre": "Pepino",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 0.70,
    "demanda_exportacion": "Media",
    "certificacion_eco": false
  },
  {
    "codigo": "MEL",
    "nombre": "Melocotón",
    "familia": "Drupa",
    "precio_mercado_euro_kg": 1.35,
    "demanda_exportacion": "Media",
    "certificacion_eco": false
  }
]
```



---

## ❓ Tareas del analista



### Tarea 1 — Inicialización del entorno

Prepara la sesión de Spark. Es el primer paso obligatorio antes de cualquier otra tarea.

**Lo que debes hacer:**

- Importar las dependencias de Spark 4.1.1 con `$ivy`
- Crear la `SparkSession` con `appName` `"AgroData_Analisis"` en modo `local[*]`
- Importar `spark.implicits._` y `org.apache.spark.sql.types._`
- Configurar el nivel de log a `ERROR`
- Imprimir un mensaje de confirmación con la versión de Spark

**Salida esperada:**

```
✅ AgroData Analytics iniciado — Spark 4.1.1
```



In [46]:
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.WARN)
Logger.getLogger("akka").setLevel(Level.WARN)

import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("AgroData_Analisis")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._   // necesario para .toDF() y otras conversiones

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ AgroData Analytics iniciado - Spark ${spark.version} listo")

26/04/28 14:44:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ AgroData Analytics iniciado - Spark 4.1.1 listo


import org.apache.log4j.{Level, Logger}
import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@64916684
import spark.implicits._

---

### Tarea 2 — Carga del CSV de parcelas con `inferSchema`

El responsable de datos quiere ver rápidamente cómo ha inferido Spark los tipos de cada columna antes de decidir si necesita un schema manual.

**Lo que debes hacer:**

- Cargar `parcelas.csv` con `header = true` e `inferSchema = true`
- Mostrar las primeras 5 filas con `show(5)`
- Imprimir el schema completo con `printSchema()`
- Imprimir el número total de parcelas registradas con `count()`
- Listar los nombres de todas las columnas con `columns`

**Salida esperada (parcial):**

```
=== Primeras 5 parcelas ===
+----------+---------+--------+-------------+-------+--------+-------------+------------------+
|id_parcela|provincia|municipio|superficie_ha|cultivo|año_alta|en_produccion|rendimiento_kg_ha|
+----------+---------+--------+-------------+-------+--------+-------------+------------------+
|      P001|  Sevilla| Carmona|         12.5| Naranja|    2015|         true|             28000|
...

=== Schema inferido ===
root
 |-- id_parcela: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- superficie_ha: double (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- año_alta: integer (nullable = true)
 |-- en_produccion: boolean (nullable = true)
 |-- rendimiento_kg_ha: integer (nullable = true)

Total de parcelas registradas: 15
Columnas: id_parcela | provincia | municipio | superficie_ha | cultivo | año_alta | en_produccion | rendimiento_kg_ha
```



In [48]:
val dfParcelas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("./datos/agrodata/parcelas.csv")

dfParcelas.show(5)
dfParcelas.printSchema()

+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|id_parcela|provincia|municipio|superficie_ha| cultivo|año_alta|en_produccion|rendimiento_kg_ha|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|      P001|  Sevilla|  Carmona|         12.5| Naranja|    2015|         true|            28000|
|      P002|   Huelva|     Lepe|          8.3|   Fresa|    2018|         true|            45000|
|      P003|  Almería|    Níjar|         25.0|  Tomate|    2012|         true|            85000|
|      P004|  Sevilla|    Écija|          6.7|Aceituna|    2009|        false|             3200|
|      P005|   Murcia|   Totana|         15.2|   Limón|    2016|         true|            22000|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
only showing top 5 rows
root
 |-- id_parcela: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio:

dfParcelas: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

---

### Tarea 3 — Definir el schema manualmente y recargar

El responsable ha revisado el schema inferido y señala dos problemas:

1. `superficie_ha` debería ser `DoubleType` — ✅ correcto tal cual
2. `rendimiento_kg_ha` debería ser `DoubleType` (no `IntegerType`) para poder hacer cálculos de medias con decimales
3. `año_alta` debería ser `IntegerType` — ✅ correcto tal cual
4. `id_parcela` es un código alfanumérico que nunca debería ser nulo: `nullable = false`

**Lo que debes hacer:**

- Definir un `StructType` con los ocho campos corrigiendo los puntos anteriores
- Recargar el CSV usando `.schema(...)` en lugar de `inferSchema`
- Imprimir el nuevo schema con `printSchema()`
- Imprimir los tipos con `dtypes` en formato `columna → tipo` para que la dirección pueda comparar fácilmente

**Salida esperada:**

```
=== Schema manual aplicado ===
root
 |-- id_parcela: string (nullable = false)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- superficie_ha: double (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- año_alta: integer (nullable = true)
 |-- en_produccion: boolean (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)

=== Tipos por columna ===
  id_parcela        → StringType
  provincia         → StringType
  municipio         → StringType
  superficie_ha     → DoubleType
  cultivo           → StringType
  año_alta          → IntegerType
  en_produccion     → BooleanType
  rendimiento_kg_ha → DoubleType
```



In [49]:
val schemaParcelas = StructType(List(
    StructField("id_parcela",      StringType,  nullable = false),
    StructField("provincia",       StringType,  nullable = true),
    StructField("municipio",       StringType,  nullable = true),
    StructField("superficie_ha",   DoubleType,  nullable = true),
    StructField("cultivo",         StringType,  nullable = true),
    StructField("año_alta",        IntegerType, nullable = true),
    StructField("en_produccion",   BooleanType, nullable = true),
    StructField("rendimiento_kg_ha", DoubleType, nullable = true)
))

val dfParcelasTyped = spark.read
    .option("header", "true")
    .schema(schemaParcelas)
    .csv("./datos/agrodata/parcelas.csv")

println("=== Schema manual aplicado ===")
dfParcelasTyped.printSchema()

println("\n=== Tipos por columna ===")
dfParcelasTyped.dtypes.foreach { case (col, tipo) =>
    println(f"  $col%-18s → $tipo")
}

=== Schema manual aplicado ===
root
 |-- id_parcela: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- superficie_ha: double (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- año_alta: integer (nullable = true)
 |-- en_produccion: boolean (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)


=== Tipos por columna ===
  id_parcela         → StringType
  provincia          → StringType
  municipio          → StringType
  superficie_ha      → DoubleType
  cultivo            → StringType
  año_alta           → IntegerType
  en_produccion      → BooleanType
  rendimiento_kg_ha  → DoubleType


schemaParcelas: StructType = Seq(
  StructField(
    name = "id_parcela",
    dataType = StringType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "provincia",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "municipio",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "superficie_ha",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "cultivo",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "año_alta",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "en_produccion",
...
dfParcelasTyped: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

---

### Tarea 4 — Primer informe estadístico de las parcelas

La directora de operaciones necesita un resumen numérico rápido de las parcelas para la reunión de mañana.

**Lo que debes hacer:**

- Usar `describe()` sobre las columnas `superficie_ha` y `rendimiento_kg_ha`
- Usar `describe()` también sobre `año_alta`
- Imprimir los resultados con un título descriptivo

**Salida esperada:**

```
=== Estadísticas de superficie y rendimiento ===
+-------+------------------+------------------+
|summary|     superficie_ha| rendimiento_kg_ha|
+-------+------------------+------------------+
|  count|                15|                15|
|   mean|   15.546666...   |        40313.3...|
| stddev|    7.54...       |        26371.4...|
|    min|               5.2|            2900.0|
|    max|              30.1|           88000.0|
+-------+------------------+------------------+

=== Rango de años de alta ===
+-------+--------+
|summary|año_alta|
+-------+--------+
|  count|      15|
|   mean|  2014.6|
| stddev|   3.26..|
|    min|    2009|
|    max|    2020|
+-------+--------+
```



In [53]:
dfParcelasTyped.describe("superficie_ha", "rendimiento_kg_ha").show()
dfParcelasTyped.describe("año_alta").show()

+-------+------------------+------------------+
|summary|     superficie_ha| rendimiento_kg_ha|
+-------+------------------+------------------+
|  count|                15|                15|
|   mean|15.479999999999999|40873.333333333336|
| stddev| 7.931240220077275|27911.479120008433|
|    min|               5.2|            2900.0|
|    max|              30.1|           88000.0|
+-------+------------------+------------------+

+-------+------------------+
|summary|          año_alta|
+-------+------------------+
|  count|                15|
|   mean|2014.5333333333333|
| stddev|3.4613512362879963|
|    min|              2009|
|    max|              2020|
+-------+------------------+



---

### Tarea 5 — Carga del catálogo de productos en JSON

El equipo comercial ha entregado el catálogo de productos en formato JSON. Hay que ingerirlo y verificar que Spark lo interpreta correctamente.

**Lo que debes hacer:**

- Cargar `productos.json` con la opción `multiline = true`
- Mostrar todos los productos con `show(truncate = false)`
- Imprimir el schema con `printSchema()`
- Anotar en un comentario del código qué tipo ha inferido Spark para `precio_mercado_euro_kg` y para `certificacion_eco`, y si te parecen correctos

**Salida esperada:**

```
=== Catálogo de productos ===
+------+---------+----------+-----------------------+-------------------+----------------+
|codigo|  nombre |  familia |precio_mercado_euro_kg |demanda_exportacion|certificacion_eco|
+------+---------+----------+-----------------------+-------------------+----------------+
|   NAR|  Naranja|   Citrico|                   0.45|               Alta|           false|
|   FRE|    Fresa|      Baya|                   2.10|           Muy Alta|            true|
...
+------+---------+----------+-----------------------+-------------------+----------------+

=== Schema del JSON ===
root
 |-- certificacion_eco: boolean (nullable = true)
 |-- codigo: string (nullable = true)
 |-- demanda_exportacion: string (nullable = true)
 |-- familia: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- precio_mercado_euro_kg: double (nullable = true)
```

> ⚠️ **Atención:** observa que Spark ordena las columnas del JSON alfabéticamente, no en el orden en que aparecen en el fichero. Esto es comportamiento normal del lector JSON de Spark.
> 



In [55]:
val dfProductosJSON = spark.read
    .option("multiline", "true")
    .json("./datos/agrodata/productos.json")

println("=== Catálogo de productos ===")
dfProductosJSON.show(truncate = false)

println("=== Schema del JSON ===")
dfProductosJSON.printSchema()

// Spark ha inferido:
// - precio_mercado_euro_kg: DoubleType
// - certificacion_eco: BooleanType
// Ambos tipos son correctos para los datos del catálogo.

=== Catálogo de productos ===
+-----------------+------+-------------------+----------+---------+----------------------+
|certificacion_eco|codigo|demanda_exportacion|familia   |nombre   |precio_mercado_euro_kg|
+-----------------+------+-------------------+----------+---------+----------------------+
|false            |NAR   |Alta               |Citrico   |Naranja  |0.45                  |
|true             |FRE   |Muy Alta           |Baya      |Fresa    |2.1                   |
|false            |TOM   |Alta               |Hortaliza |Tomate   |0.85                  |
|true             |ACE   |Media              |Oleaginosa|Aceituna |0.6                   |
|false            |LIM   |Alta               |Citrico   |Limón    |0.55                  |
|true             |PIM   |Muy Alta           |Hortaliza |Pimiento |1.2                   |
|false            |PEP   |Media              |Hortaliza |Pepino   |0.7                   |
|false            |MEL   |Media              |Drupa     |Mel

dfProductosJSON: org.apache.spark.sql.package.DataFrame = [certificacion_eco: boolean, codigo: string ... 4 more fields]

---

### Tarea 6 — Schema manual para el catálogo de productos

El equipo comercial indica que `codigo` nunca debería ser nulo (es la clave del producto) y que `precio_mercado_euro_kg` debe ser siempre `DoubleType` con `nullable = false`.

**Lo que debes hacer:**

- Definir un `StructType` para los seis campos del JSON respetando las restricciones anteriores
- Recargar el JSON con el schema manual
- Comparar los tipos resultantes con los de la carga por inferencia usando `dtypes`



---

### Tarea 7 — DataFrame de resumen desde colección en memoria

El director general ha pedido que se incluya en el informe una pequeña tabla resumen creada a mano con los datos más importantes de cada provincia, antes de tener el análisis completo automatizado.

**Lo que debes hacer:**

- Crear el siguiente DataFrame **desde una colección Scala** usando `.toDF(...)`:

| provincia | num_parcelas | superficie_total_ha | parcelas_activas |
| --- | --- | --- | --- |
| Almería | 4 | 95.4 | 4 |
| Huelva | 3 | 32.4 | 3 |
| Murcia | 4 | 58.0 | 3 |
| Sevilla | 4 | 46.4 | 2 |
- Mostrar el DataFrame con `show()`
- Imprimir su schema con `printSchema()`
- Confirmar que Spark ha inferido los tipos correctamente para cada columna
- Imprimir el número de filas con `count()` y los nombres de columnas con `columns`

**Salida esperada del schema:**

```
root
 |-- provincia: string (nullable = true)
 |-- num_parcelas: integer (nullable = false)
 |-- superficie_total_ha: double (nullable = false)
 |-- parcelas_activas: integer (nullable = false)
```



---

## 🗺️ Guía de herramientas → tareas

| Tarea | Herramientas principales |
| --- | --- |
| 1 — Inicialización | `SparkSession.builder`, `$ivy`, `import spark.implicits._` |
| 2 — CSV con inferSchema | `spark.read.option(...).csv(...)`, `show`, `printSchema`, `count`, `columns` |
| 3 — Schema manual CSV | `StructType`, `StructField`, `DoubleType`, `BooleanType`, `.schema(...)`, `dtypes` |
| 4 — Estadísticas | `describe(...)` |
| 5 — JSON con inferSchema | `spark.read.option("multiline","true").json(...)`, `show(truncate=false)`, `printSchema` |
| 6 — Schema manual JSON | `StructType`, `StructField`, `.schema(...)`, `dtypes` |
| 7 — Colección en memoria | `Seq(...).toDF(...)`, `show`, `printSchema`, `count`, `columns` |

---

<aside>



## 💡 Consejo final

En este caso no hay transformaciones ni filtros: el objetivo es **conocer los datos antes de tocarlos**. En la práctica profesional, el primer trabajo de un analista al recibir un fichero nuevo siempre es:

1. Cargarlo
2. Revisar el schema
3. Contar filas
4. Ver estadísticas básicas
5. Anotar anomalías (tipos inesperados, nulos, rangos extraños)

Todo lo que has practicado hoy es exactamente ese flujo. Las transformaciones vienen después. 

</aside>